In [ ]:
"""
FASE 4 - Paso 2 (tres casos): M(d) en escala log-log
=====================================================
Tesis: Complejidad e informacion en un minority game con opcion de abstencion

Que hace este codigo
--------------------
NO vuelve a simular. Lee los CSV "todas_las_cadenas" que genero el codigo
Segundo_paso_tres_casos (N = 50, m = 3, K = 3, T = 6000, 100 semillas) y,
para cada uno de los tres casos:

  1. Promedia M(d) sobre las 100 cadenas, para d = 1, ..., 100
     (es la misma curva promedio que se grafico en el Paso 2).
  2. Quita los puntos con M(d) = 0, porque log(0) no existe.
     (Es el UNICO filtro, por indicacion del Dr. Mansilla.)
  3. Calcula log10(d) y log10(M(d)).
  4. Ajusta una recta por minimos cuadrados con TODOS los puntos que quedan:
         log10 M(d) = b + pendiente * log10(d)
     Si M(d) = A / d^alpha, entonces pendiente = -alpha y b = log10(A).
  5. Reporta la PENDIENTE y la R^2 del ajuste.
  6. Grafica log10 M(d) contra log10 d con la recta ajustada.

Notacion: gamma = 3^m / N es el parametro de control; alpha es el
exponente de la ley de potencias M(d) = A / d^alpha.
"""

import numpy as np
import matplotlib.pyplot as plt


# =====================================================================
# OPCIONES  <-- AQUI SE DECIDE SI SE GUARDAN LOS ARCHIVOS
# =====================================================================
# Las graficas SIEMPRE se muestran en el notebook.
# True  = ademas se guardan como PNG en la carpeta CARPETA_SALIDA
# False = solo se muestran, no se guarda ningun PNG
DESCARGAR_GRAFICAS = True

# True  = se guarda un CSV con la pendiente y la R^2 de cada caso
GUARDAR_CSV        = True

CARPETA_SALIDA = "resultados_fase4_paso1"


# =====================================================================
# DATOS DE ENTRADA (los CSV del codigo Segundo_paso_tres_casos)
# =====================================================================
N, M, K, T, N_CADENAS = 50, 3, 3, 6000, 100
GAMMA = K ** M / N     # parametro de control 3^m / N = 0.54

CASOS = [
    {"nombre": "ingenuos",
     "titulo": "agentes ingenuos",
     "csv":    "Md_ingenuos_todas_las_cadenas.csv",
     "png":    "loglog_Md_ingenuos.png"},
    {"nombre": "2 estrategias sin puntaje",
     "titulo": "2 estrategias, sin puntaje",
     "csv":    "Md_estrategias_sin_recompensa_todas_las_cadenas.csv",
     "png":    "loglog_Md_estrategias_sin_puntaje.png"},
    {"nombre": "2 estrategias con puntaje",
     "titulo": "2 estrategias, con puntaje",
     "csv":    "Md_2estrategias_con_puntaje_todas_las_cadenas.csv",
     "png":    "loglog_Md_2estrategias_con_puntaje.png"},
]


# =====================================================================
# LECTURA Y PROMEDIO
# =====================================================================
def leer_promedio(archivo):
    """Lee el CSV de todas las cadenas y devuelve d y el promedio de M(d).

    Formato del CSV: 3 lineas de encabezado que empiezan con '#';
    despues, una fila por semilla: semilla, M(d=1), ..., M(d=100).
    """
    tabla = np.loadtxt(archivo, delimiter=",", comments="#")
    valores = tabla[:, 1:]                        # se quita la columna semilla
    distancias = np.arange(1, valores.shape[1] + 1)
    return distancias, valores.mean(axis=0), valores.shape[0]


# =====================================================================
# AJUSTE LINEAL EN ESCALA LOG-LOG
# =====================================================================
def ajuste_loglog(d, Md):
    """Quita los ceros, pasa a log10 y ajusta una recta.

    Devuelve un diccionario con la pendiente, la ordenada al origen,
    la R^2, los puntos usados y cuantos ceros se quitaron.
    """
    validos = Md > 0                      # UNICO filtro: fuera M(d) = 0
    x = np.log10(d[validos])
    y = np.log10(Md[validos])

    pendiente, ordenada = np.polyfit(x, y, 1)   # recta y = ordenada + pendiente x
    y_ajuste = ordenada + pendiente * x

    ss_res = np.sum((y - y_ajuste) ** 2)        # suma de cuadrados residual
    ss_tot = np.sum((y - y.mean()) ** 2)        # suma de cuadrados total
    r2 = 1 - ss_res / ss_tot

    return {
        "pendiente": pendiente,
        "ordenada": ordenada,
        "r2": r2,
        "x": x, "y": y, "y_ajuste": y_ajuste,
        "n_puntos": int(validos.sum()),
        "n_ceros": int((~validos).sum()),
    }


# =====================================================================
# GRAFICA
# =====================================================================
AZUL = "#2a78d6"     # mismo azul de las graficas del Paso 2
GRIS = "#52514e"
TINTA = "#0b0b0b"
REJILLA = "#e5e4df"

plt.rcParams.update({
    "font.size": 11,
    "axes.edgecolor": GRIS,
    "axes.labelcolor": TINTA,
    "xtick.color": GRIS,
    "ytick.color": GRIS,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


def graficar(caso, res, guardar):
    fig, ax = plt.subplots(figsize=(9, 5))

    ax.plot(res["x"], res["y"], "o", color=AZUL, ms=5,
            label=f"log$_{{10}}$ M(d), promedio de {N_CADENAS} cadenas")
    ax.plot(res["x"], res["y_ajuste"], color=TINTA, lw=2,
            label="Ajuste lineal")

    # resultados del ajuste dentro de la grafica
    texto = (f"pendiente = {res['pendiente']:.4f}\n"
             f"R$^2$ = {res['r2']:.4f}")
    ax.text(0.98, 0.95, texto, transform=ax.transAxes, ha="right", va="top",
            fontsize=11, color=TINTA,
            bbox=dict(boxstyle="round,pad=0.4", fc="white", ec=REJILLA))

    ax.set_xlabel("log$_{10}$ d")
    ax.set_ylabel("log$_{10}$ M(d)   [M(d) en bits]")
    ax.set_title(f"M(d) en escala log-log — {caso['titulo']}\n"
                 f"N={N}, m={M}, K={K}, T={T}, γ = 3$^m$/N = {GAMMA:.2f}, "
                 f"{N_CADENAS} semillas",
                 loc="left", color=TINTA)
    ax.grid(color=REJILLA, lw=0.8)
    ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.14),
              ncol=2, fontsize=9)
    fig.tight_layout()

    if guardar:
        import os
        os.makedirs(CARPETA_SALIDA, exist_ok=True)
        ruta = os.path.join(CARPETA_SALIDA, caso["png"])
        fig.savefig(ruta, dpi=200)
        print(f"  Guardado: {ruta}")

    plt.show()
    plt.close(fig)


# =====================================================================
# PROGRAMA PRINCIPAL
# =====================================================================
if __name__ == "__main__":

    resumen = []

    for caso in CASOS:
        d, Md, n_cad = leer_promedio(caso["csv"])
        res = ajuste_loglog(d, Md)

        print("\n" + "=" * 62)
        print(f"  {caso['titulo']}   ({n_cad} cadenas, d = 1..{d[-1]})")
        print("=" * 62)
        print(f"  Puntos con M(d) = 0 eliminados: {res['n_ceros']}")
        print(f"  Puntos usados en el ajuste:     {res['n_puntos']}")
        print(f"  Pendiente:                      {res['pendiente']:.4f}")
        print(f"  R^2:                            {res['r2']:.4f}")

        graficar(caso, res, DESCARGAR_GRAFICAS)

        resumen.append([caso["nombre"], res["n_ceros"], res["n_puntos"],
                        res["pendiente"], res["r2"], res["ordenada"]])

    # --- tabla resumen ---
    print("\n" + "=" * 62)
    print("  RESUMEN  (N=50, m=3, T=6000, gamma=0.54)")
    print("=" * 62)
    print(f"  {'caso':28s} {'pendiente':>10s} {'R^2':>8s}")
    for nombre, _, _, p, r2, _ in resumen:
        print(f"  {nombre:28s} {p:>10.4f} {r2:>8.4f}")

    if GUARDAR_CSV:
        import os
        os.makedirs(CARPETA_SALIDA, exist_ok=True)
        ruta = os.path.join(CARPETA_SALIDA, "ajustes_loglog_tres_casos.csv")
        with open(ruta, "w", encoding="utf-8") as f:
            f.write("# Ajuste lineal de log10 M(d) vs log10 d, d = 1..100, "
                    "promedio de 100 cadenas\n")
            f.write(f"# N={N} m={M} K={K} T={T} gamma={GAMMA:.2f}; "
                    "unico filtro: se quitan los M(d) = 0\n")
            f.write("caso,ceros_eliminados,puntos_usados,pendiente,R2,"
                    "ordenada_log10\n")
            for nombre, c, n, p, r2, b in resumen:
                f.write(f"{nombre},{c},{n},{p:.6f},{r2:.6f},{b:.6f}\n")
        print(f"\n  Guardado: {ruta}")